# RAG Inspection Notebook

Этот notebook нужен для учебной инспекции RAG-пайплайна проекта `cat-breed-assistant`.

Цель: пошагово проверить цепочку:

```text
Hugging Face dataset → preprocessing → chunks → embeddings → ChromaDB → retrieval → LLM answer
```

Notebook помогает увидеть, какие chunks реально попадают в retrieval и дальше в prompt для LLM.

## Setup

Импортируем библиотеки, добавляем root проекта в `sys.path`, подключаем проектные функции и проверяем ключевые пути.

In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import sys
from pathlib import Path
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.breed_retriever import build_breed_context
from src.cat_knowledge import generate_mock_answer
from src.mistral_client import generate_mistral_answer
from src.rag.preprocess import (
    SOURCE_DATASET,
    extract_assistant_chunks,
    extract_breed_from_image_path,
    extract_image_path,
    read_jsonl,
)
from src.rag.retriever import retrieve_relevant_chunks
from src.rag.vector_store import CHROMA_PATH, COLLECTION_NAME, get_chroma_client, get_existing_collection

load_dotenv(PROJECT_ROOT / ".env")

PROCESSED_CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "rag_chunks.jsonl"
CHROMA_DIR = PROJECT_ROOT / "data" / "chroma"
BREED_PROFILES_PATH = PROJECT_ROOT / "data" / "breed_profiles.json"

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())

for path in [PROCESSED_CHUNKS_PATH, CHROMA_DIR, BREED_PROFILES_PATH]:
    print(f"{path.relative_to(PROJECT_ROOT)} exists:", path.exists())

## Load Hugging Face Dataset

Загружаем `YZhao09/cat_breed_meowticulous`, split `train`, и смотрим на сырые поля. Если датасет недоступен, ячейка покажет понятное сообщение и notebook можно продолжать с уже обработанными chunks.

In [ ]:
try:
    from datasets import load_dataset

    dataset = load_dataset(SOURCE_DATASET, split="train")
    print("Rows:", len(dataset))
    print("Columns:", dataset.column_names)

    compact_rows = []
    for index in range(min(2, len(dataset))):
        row = dataset[index]
        compact_rows.append(
            {
                "id": row.get("id", index),
                "image_path": extract_image_path(row.get("image")),
                "conversation_turns": len(row.get("conversations") or []),
            }
        )
    display(pd.DataFrame(compact_rows))

    sample = dataset[0]
    print("Example image:")
    print(extract_image_path(sample.get("image")))
    print("\nExample conversations:")
    pprint(sample.get("conversations"))
except Exception as error:
    dataset = None
    print("Не удалось загрузить Hugging Face dataset.")
    print("Причина:", error)
    print("Можно продолжить notebook, если уже есть data/processed/rag_chunks.jsonl.")
    print("Для подготовки данных запустите:")
    print("python scripts/download_hf_dataset.py")
    print("python scripts/build_rag_index.py")

## Inspect Raw Conversations

Посмотрим несколько сырых примеров: id, путь картинки, извлечённую породу, human question и assistant answer.

In [ ]:
def conversation_text(turn: dict) -> str:
    for key in ("value", "content", "text", "message"):
        value = turn.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""


def role_of(turn: dict) -> str:
    return str(turn.get("from") or turn.get("role") or turn.get("speaker") or "").casefold()


def first_human_and_assistant(conversations: list[dict]) -> tuple[str, str]:
    human = ""
    assistant = ""

    for index, turn in enumerate(conversations or []):
        role = role_of(turn)
        text = conversation_text(turn)
        if not text:
            continue
        if not human and (role in {"human", "user"} or index % 2 == 0):
            human = text
        elif not assistant and (role in {"assistant", "gpt", "model", "bot"} or index % 2 == 1):
            assistant = text
        if human and assistant:
            break

    return human, assistant

if dataset is None:
    print("Dataset не загружен, пропускаем raw inspection.")
else:
    rows = []
    for index in range(min(5, len(dataset))):
        example = dataset[index]
        image_path = extract_image_path(example.get("image"))
        human, assistant = first_human_and_assistant(example.get("conversations") or [])
        rows.append(
            {
                "id": example.get("id", index),
                "image_path": image_path,
                "breed": extract_breed_from_image_path(image_path),
                "human_question": human[:180],
                "assistant_answer": assistant[:240],
            }
        )
    display(pd.DataFrame(rows))

## Run Preprocessing Step By Step

Здесь не вызываем внешний скрипт целиком. Показываем, как из `image path` получается breed, как из `conversations` достаются assistant-ответы и как формируются `chunk_id` и `metadata`.

In [ ]:
if dataset is None:
    print("Dataset не загружен. Для этой ячейки нужен доступ к Hugging Face dataset.")
else:
    sample_chunks = []
    for row_index in range(min(10, len(dataset))):
        example = dataset[row_index]
        image_path = extract_image_path(example.get("image"))
        breed = extract_breed_from_image_path(image_path)
        chunks = extract_assistant_chunks(example, split="train", row_index=row_index)

        print("Example", row_index)
        print("  image_path:", image_path)
        print("  extracted breed:", breed)
        print("  assistant chunks:", len(chunks))

        sample_chunks.extend(chunks)

    df = pd.DataFrame(
        [
            {
                "chunk_id": chunk["chunk_id"],
                "breed": chunk["metadata"]["breed"],
                "text_preview": chunk["text"][:180],
                "source_id": chunk["metadata"]["source_id"],
                "image_path": chunk["metadata"]["image_path"],
                "turn_index": chunk["metadata"]["turn_index"],
            }
            for chunk in sample_chunks
        ]
    )
    display(df.head(20))

## Load Processed Chunks

Загружаем `data/processed/rag_chunks.jsonl` и смотрим распределение chunks по породам, случайные примеры и длины текстов.

In [ ]:
if not PROCESSED_CHUNKS_PATH.exists():
    chunks = []
    print("Файл processed chunks не найден:", PROCESSED_CHUNKS_PATH)
    print("Подготовьте RAG data командами:")
    print("python scripts/download_hf_dataset.py")
    print("python scripts/build_rag_index.py")
else:
    chunks = read_jsonl(PROCESSED_CHUNKS_PATH)
    chunk_df = pd.DataFrame(
        [
            {
                "chunk_id": chunk["chunk_id"],
                "breed": chunk["metadata"].get("breed"),
                "text": chunk["text"],
                "text_len": len(chunk["text"]),
                "source_id": chunk["metadata"].get("source_id"),
                "image_path": chunk["metadata"].get("image_path"),
                "turn_index": chunk["metadata"].get("turn_index"),
            }
            for chunk in chunks
        ]
    )

    print("Chunks loaded:", len(chunk_df))
    print("Unique breeds:", chunk_df["breed"].nunique())
    print("Chunk length min / mean / max:", chunk_df["text_len"].min(), round(chunk_df["text_len"].mean(), 1), chunk_df["text_len"].max())

    display(chunk_df["breed"].value_counts().head(20).rename_axis("breed").reset_index(name="chunks"))
    display(chunk_df.sample(min(5, len(chunk_df)), random_state=42)[["chunk_id", "breed", "text", "source_id"]])

## Embedding Model Sanity Check

Embeddings превращают текст в числовые векторы. Затем можно искать chunks, чей смысл ближе к смыслу запроса. Ниже считаем embeddings для нескольких chunks и сравниваем cosine similarity с запросом.

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
QUERY = "What cat breed has a round face and dense plush coat?"

if not chunks:
    print("Нет chunks для embedding sanity check.")
else:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity

        model = SentenceTransformer(MODEL_NAME, local_files_only=True)
        sample_texts = [chunk["text"] for chunk in chunks[:5]]
        embeddings = model.encode(sample_texts, convert_to_numpy=True)
        query_embedding = model.encode([QUERY], convert_to_numpy=True)
        similarities = cosine_similarity(query_embedding, embeddings)[0]

        print("Embedding shape:", embeddings.shape)
        display(
            pd.DataFrame(
                {
                    "text_preview": [text[:180] for text in sample_texts],
                    "cosine_similarity": similarities,
                }
            ).sort_values("cosine_similarity", ascending=False)
        )
    except Exception as error:
        print("Не удалось загрузить embedding model локально.")
        print("Причина:", error)
        print("Если модель ещё не скачана, запустите: python scripts/build_rag_index.py")

## Inspect ChromaDB Index

Подключаемся к локальному ChromaDB index в `data/chroma/` и смотрим collection `cat_breed_chunks`.

In [ ]:
try:
    client = get_chroma_client(CHROMA_DIR)
    collections = client.list_collections()
    collection_names = [getattr(collection, "name", collection) for collection in collections]
    print("Collections:", collection_names)

    collection = get_existing_collection(client)
    if collection is None:
        print("Collection не найдена. Запустите: python scripts/build_rag_index.py")
    else:
        print("Documents in collection:", collection.count())
        sample_records = collection.peek(limit=5)
        display(
            pd.DataFrame(
                [
                    {
                        "id": sample_records.get("ids", [])[index],
                        "breed": sample_records.get("metadatas", [])[index].get("breed"),
                        "source_id": sample_records.get("metadatas", [])[index].get("source_id"),
                        "text_preview": sample_records.get("documents", [])[index][:180],
                    }
                    for index in range(len(sample_records.get("ids", [])))
                ]
            )
        )
except Exception as error:
    collection = None
    print("Не удалось подключиться к ChromaDB index.")
    print("Причина:", error)
    print("Создайте index командой: python scripts/build_rag_index.py")

## Manual Retrieval

Делаем retrieval напрямую через ChromaDB. Это полезно, чтобы отделить качество vector search от логики backend/provider.

In [ ]:
MANUAL_QUERIES = [
    "What cat has a round face and dense coat?",
    "Tell me about Maine Coon temperament",
    "Какая кошка известна круглой мордой и плотной шерстью?",
    "Чем мейн-кун отличается от перса?",
]


def manual_query(query: str, top_k: int = 5, breed: str | None = None) -> pd.DataFrame:
    if collection is None:
        return pd.DataFrame()

    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(MODEL_NAME, local_files_only=True)
    query_embedding = model.encode([query], convert_to_numpy=True)[0].tolist()
    kwargs = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if breed:
        kwargs["where"] = {"breed": breed}

    result = collection.query(**kwargs)
    rows = []
    for rank, (text, metadata, distance) in enumerate(
        zip(result["documents"][0], result["metadatas"][0], result["distances"][0]),
        start=1,
    ):
        rows.append(
            {
                "rank": rank,
                "distance": distance,
                "breed": metadata.get("breed"),
                "source_id": metadata.get("source_id"),
                "text_preview": text[:220],
            }
        )
    return pd.DataFrame(rows)

for query in MANUAL_QUERIES:
    print("QUERY:", query)
    display(manual_query(query))

## Project Retriever Check

Теперь используем проектную функцию `retrieve_relevant_chunks(question, top_k=5)`. Также явно смотрим detected breed из `build_breed_context`, потому что backend может передавать breed в metadata-фильтр.

In [ ]:
for query in MANUAL_QUERIES:
    breed_context = build_breed_context(query)
    detected_breed = breed_context["breed"]
    chunks_without_filter = retrieve_relevant_chunks(query, top_k=5)
    chunks_with_filter = retrieve_relevant_chunks(query, top_k=5, breed=detected_breed)

    print("QUERY:", query)
    print("Detected breed:", detected_breed)
    print("Without explicit breed filter:", [chunk["metadata"].get("breed") for chunk in chunks_without_filter])
    print("With detected breed filter:", [chunk["metadata"].get("breed") for chunk in chunks_with_filter])

    display(
        pd.DataFrame(
            [
                {
                    "rank": index,
                    "distance": chunk.get("distance"),
                    "breed": chunk.get("metadata", {}).get("breed"),
                    "source_id": chunk.get("metadata", {}).get("source_id"),
                    "text_preview": chunk.get("text", "")[:220],
                }
                for index, chunk in enumerate(chunks_with_filter, start=1)
            ]
        )
    )

## Debug Russian Query Problem

Короткие русские запросы могут плохо матчиться с англоязычными chunks, потому что модель `all-MiniLM-L6-v2` не является сильной multilingual моделью. Сравним русские и английские варианты.

In [ ]:
RUSSIAN_DEBUG_QUERIES = [
    "британские котики",
    "британская короткошёрстная кошка",
    "British Shorthair cat",
    "round face plush coat",
]

for query in RUSSIAN_DEBUG_QUERIES:
    breed_context = build_breed_context(query)
    detected_breed = breed_context["breed"]
    print("QUERY:", query)
    print("Detected breed:", detected_breed)
    print("Direct Chroma search:")
    display(manual_query(query))
    print("Project retriever with detected breed filter:")
    retrieved = retrieve_relevant_chunks(query, top_k=5, breed=detected_breed)
    display(
        pd.DataFrame(
            [
                {
                    "rank": index,
                    "distance": chunk.get("distance"),
                    "breed": chunk.get("metadata", {}).get("breed"),
                    "text_preview": chunk.get("text", "")[:220],
                }
                for index, chunk in enumerate(retrieved, start=1)
            ]
        )
    )

### Вывод по русским запросам

- Если английский запрос даёт более релевантные chunks, значит текущая embedding model недостаточно multilingual.
- Breed filter помогает, когда `build_breed_context` уверенно определяет породу.
- Breed filter может быть слишком агрессивным, если вопрос сравнительный или порода не распознана.
- Для улучшения можно использовать multilingual embedding model или переводить русский query на английский перед retrieval.

## Build Final RAG Prompt

Собираем пример prompt, который затем может быть отправлен в LLM: system instruction, breed context, retrieved context и вопрос пользователя.

In [ ]:
FINAL_QUESTION = "Чем британские котики отличаются от обычных?"
breed_context = build_breed_context(FINAL_QUESTION)
retrieved_context = retrieve_relevant_chunks(FINAL_QUESTION, top_k=5, breed=breed_context["breed"])

system_instruction = """
Ты дружелюбный ассистент для учебного приложения про породы кошек.
Отвечай на русском языке понятно для обычного пользователя.
Используй retrieved context как основной источник фактов.
Не выдумывай факты, которых нет в контексте.
Если данных мало, честно скажи, что информации мало.
Не давай категоричных медицинских советов; при вопросах о здоровье советуй обратиться к ветеринару.
""".strip()

prompt = {
    "system_instruction": system_instruction,
    "user_question": FINAL_QUESTION,
    "breed_context": breed_context,
    "retrieved_context": retrieved_context,
}

print(json.dumps(prompt, ensure_ascii=False, indent=2)[:8000])

## LLM Answer Test

Если в `.env` есть `MISTRAL_API_KEY`, проверяем live Mistral provider. Если ключа нет, не падаем и показываем mock answer using retrieved chunks.

In [ ]:
if os.getenv("MISTRAL_API_KEY"):
    print("MISTRAL_API_KEY найден. Вызываем Mistral provider без печати ключа.")
    try:
        answer = generate_mistral_answer(FINAL_QUESTION, breed_context, retrieved_context)
        print(answer)
    except Exception as error:
        print("Mistral call failed:", error)
        print("Fallback mock answer:")
        print(generate_mock_answer(FINAL_QUESTION, breed_context, retrieved_context))
else:
    print("MISTRAL_API_KEY not found, skipping live LLM call")
    print("Mock answer using retrieved chunks:")
    print(generate_mock_answer(FINAL_QUESTION, breed_context, retrieved_context))

## Backend Integration Check

Проверяем FastAPI endpoint через `TestClient`, без запуска `uvicorn`.

In [ ]:
from fastapi.testclient import TestClient
from backend.main import app

client = TestClient(app)
payload = {
    "question": "Чем британские котики отличаются от обычных?",
    "mode": "mock",
    "use_rag": True,
}

response = client.post("/ask", json=payload)
data = response.json()

print("status_code:", response.status_code)
print("rag_enabled:", data.get("rag_enabled"))
print("breed:", data.get("breed"))
print("mode:", data.get("mode"))
print("retrieved_context count:", len(data.get("retrieved_context", [])))
print("\nAnswer preview:")
print((data.get("answer") or "")[:700])

print("\nRetrieved chunks preview:")
for index, chunk in enumerate(data.get("retrieved_context", [])[:5], start=1):
    metadata = chunk.get("metadata", {})
    print(f"{index}. breed={metadata.get('breed')} source_id={metadata.get('source_id')} distance={chunk.get('distance')}")
    print(chunk.get("text", "")[:260])
    print("---")

## Conclusions

Заполните после запуска notebook:

- RAG index built / not built: смотрим `data/chroma/` и `cat_breed_chunks` count.
- Processed chunks count: смотрим `data/processed/rag_chunks.jsonl`.
- Chroma docs count: смотрим `collection.count()`.
- Хорошо обрабатываются запросы, где порода явно указана на английском или хорошо распознана через aliases.
- Слабее обрабатываются короткие русские запросы, если vector model плохо связывает русский query с английскими chunks.
- Breed filter помогает, но может мешать сравнительным вопросам, где нужны chunks по нескольким породам.

## Possible Improvements

- Использовать multilingual embedding model.
- Переводить русский query на английский перед retrieval.
- Улучшить breed detection и поддержку сравнительных запросов.
- Добавить hybrid retrieval: breed filter + vector search + keyword search.
- Показывать retrieved chunks в Streamlit заметнее.
- Добавить evaluation set с ожидаемыми породами и релевантными chunks.